# 7-2절 연습 문제 풀이

이 노트북은 7-2절 연습 문제(7-4 ~ 7-7)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch07/07-02_example.ipynb`를 참고한다.
- 소설 텍스트는 저장소 규약에 따라 `../../data/`에서 읽는다.
- 학습 시간을 줄이기 위해 본문(30 에포크)보다 짧게 학습하되, 비교하는 모델끼리는 조건을 같게 맞췄다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import copy
import random
import re
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'학습 장치: {device}')

DATA_DIR = '../../data'
SEQUENCE_LENGTH = 20
BATCH_SIZE = 64
EMBEDDING_DIM = 128
HIDDEN_SIZE = 128
LEARNING_RATE = 0.001
EPOCHS = 15            # 본문은 30 에포크. 비교 실험이 많아 15로 줄였다

학습 장치: cuda


In [2]:
# 본문 예제와 같은 전처리·토큰화 과정
APOS = '\u2019'          # 원문에 사용된 곱슬 아포스트로피

def tokenize(text):
    normalized = text.lower()
    normalized = re.sub(f"[^a-z\\s,.!?{APOS}']", ' ', normalized)
    tokens = re.findall(f"[a-z{APOS}']+|[,.!?]", normalized)
    tokens = [t.rstrip(APOS) if not t.endswith("'") else t for t in tokens]
    return [t for t in tokens if t]

def load_novel(path):
    with open(path, encoding='utf-8-sig') as file:
        raw = file.read()
    start = re.search(r'\*\*\* START OF .*?\*\*\*', raw)
    end = re.search(r'\*\*\* END OF .*?\*\*\*', raw)
    if start and end:
        raw = raw[start.end(): end.start()]
    return raw

oz_tokens = tokenize(load_novel(f'{DATA_DIR}/wonderful_wizard_of_oz.txt'))
print(f'<오즈의 마법사> 전체 토큰 {len(oz_tokens):,}개, 고유 토큰 {len(set(oz_tokens)):,}개')
print(f'처음 20개: {oz_tokens[:20]}')

<오즈의 마법사> 전체 토큰 44,726개, 고유 토큰 2,966개
처음 20개: ['illustration', 'the', 'wonderful', 'wizard', 'of', 'oz', 'by', 'l', '.', 'frank', 'baum', 'this', 'book', 'is', 'dedicated', 'to', 'my', 'good', 'friend', 'comrade']


In [3]:
# 본문 [코드 7-4]의 데이터셋 클래스
class OzDataset(Dataset):
    def __init__(self, token_idxs, sequence_length):
        self.token_idxs = token_idxs
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.token_idxs) - self.sequence_length

    def __getitem__(self, idx):
        X = torch.tensor(self.token_idxs[idx: idx + self.sequence_length])
        y = torch.tensor(self.token_idxs[idx + self.sequence_length])
        return X.long(), y.long()

def build_vocab(tokens):
    vocab = {t: i for i, t in enumerate(sorted(set(tokens)))}
    return vocab, {i: t for t, i in vocab.items()}

oz_vocab, oz_reversed_vocab = build_vocab(oz_tokens)
oz_idxs = [oz_vocab[t] for t in oz_tokens]
oz_dataset = OzDataset(oz_idxs, SEQUENCE_LENGTH)
oz_loader = DataLoader(oz_dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f'어휘 사전 크기: {len(oz_vocab):,}')
print(f'샘플 수: {len(oz_dataset):,}')

어휘 사전 크기: 2,966
샘플 수: 44,706


In [4]:
# 본문 [코드 7-3]의 모델과 임베딩 없는 대조군
class OzWriter(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):               # (B, S)
        x = self.embeddings(x)          # -> (B, S, D)
        x, _ = self.lstm(x)             # -> (B, S, hidden_size)
        return self.fc(x[:, -1, :])     # -> (B, V)

class OzRawWriter(nn.Module):
    """임베딩 없이 원-핫 인코딩된 입력을 LSTM에 바로 넣는 대조군 모델."""

    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.lstm = nn.LSTM(vocab_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):               # (B, S)
        x = nn.functional.one_hot(x, num_classes=self.vocab_size).float()
        x, _ = self.lstm(x)             # -> (B, S, hidden_size)
        return self.fc(x[:, -1, :])

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

## 연습 문제 7-4

> `OzWriter` 모델의 학습 루프와 문장 생성 함수를 직접 만들어 보자. 참고로 7-2절의 깃허브 예제 노트북을
> 조금만 내리면 저자가 작성한 예제 코드가 있지만, 이를 확인하기 전 5장과 6장에서 학습한 다음의 기능을
> 구현할 수 있는지 직접 테스트해 보자.
> - 지정한 CPU 또는 하드웨어 가속기 장치를 사용하는 학습과 예측
> - 순환 신경망 모델의 학습 루프
> - 정해진 수의 토큰을 생성해 문장을 만드는 생성 함수. 단, 비정상 종료되지 않도록 예외 처리가 되어 있어야 한다.

In [5]:
# 요구 1, 2 — 장치를 지정할 수 있는 1 에포크 학습 함수와 학습 루프
def train_epoch(model, loader, criterion, optimizer, device, clip=1.0):
    model.train()
    loss_sum, sample_size = 0.0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)   # 6장의 기울기 클리핑
        optimizer.step()
        loss_sum += loss.item() * inputs.size(0)
        sample_size += inputs.size(0)
    return loss_sum / sample_size

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    loss_sum, sample_size = 0.0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        loss_sum += criterion(model(inputs), labels).item() * inputs.size(0)
        sample_size += inputs.size(0)
    return loss_sum / sample_size

def train(model, train_loader, valid_loader=None, epochs=EPOCHS,
          learning_rate=LEARNING_RATE, device=device, label=''):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    history, started = [], time.time()
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_loss = validate(model, valid_loader, criterion, device) if valid_loader else float('nan')
        history.append((epoch, train_loss, valid_loss))
        if valid_loader:
            print(f'  {label}에포크 {epoch:2d} | 훈련 손실 {train_loss:.4f} | 검증 손실 {valid_loss:.4f}')
        else:
            print(f'  {label}에포크 {epoch:2d} | 훈련 손실 {train_loss:.4f}')
    return {'history': history, 'elapsed': time.time() - started}

In [6]:
# 요구 3 — 예외 처리를 갖춘 생성 함수
UNK_TOKEN = '<unk>'

@torch.no_grad()
def generate_text(model, primer, length, vocab, reversed_vocab,
                  sequence_length=SEQUENCE_LENGTH, temperature=0.0, device=device):
    """마중물 텍스트에서 시작해 정해진 수의 토큰을 생성한다.

    비정상 종료를 막기 위해 다음 세 가지를 처리한다.
      (1) 어휘 사전에 없는 토큰 -> <unk>가 있으면 그 번호로, 없으면 건너뜀
      (2) 마중물이 sequence_length보다 짧음 -> 앞을 마침표로 채움
      (3) 마중물이 비었거나 남는 토큰이 없음 -> ValueError 로 분명하게 알림
    """
    model.eval()
    tokens = tokenize(primer)
    unk_idx = vocab.get(UNK_TOKEN)
    known, unknown = [], []
    for token in tokens:
        if token in vocab:
            known.append(token)
        else:
            unknown.append(token)
            if unk_idx is not None:
                known.append(UNK_TOKEN)
    if not known:
        raise ValueError('마중물 텍스트에 어휘 사전에 있는 토큰이 하나도 없습니다.')
    if unknown:
        print(f'  (주의) 어휘 사전에 없는 토큰 {unknown} -> '
              f'{"<unk>로 대체" if unk_idx is not None else "건너뜀"}')

    result = list(known)
    for _ in range(length):
        window = result[-sequence_length:]
        if len(window) < sequence_length:              # 마중물이 짧으면 앞을 채움
            window = ['.'] * (sequence_length - len(window)) + window
        idx = torch.tensor([[vocab.get(t, unk_idx or 0) for t in window]]).long().to(device)
        logits = model(idx)[0]
        if temperature <= 0:
            next_idx = logits.argmax().item()
        else:
            probabilities = torch.softmax(logits / temperature, dim=0)
            next_idx = torch.multinomial(probabilities, 1).item()
        result.append(reversed_vocab[next_idx])
    return ' '.join(result)

print('생성 함수 준비 완료')

생성 함수 준비 완료


In [7]:
torch.manual_seed(SEED)
oz_writer = OzWriter(len(oz_vocab), EMBEDDING_DIM, HIDDEN_SIZE)
print(f'OzWriter 파라미터 수: {count_parameters(oz_writer):,}')
print('OzWriter 학습')
oz_result = train(oz_writer, oz_loader)

OzWriter 파라미터 수: 894,358
OzWriter 학습


  에포크  1 | 훈련 손실 5.5581


  에포크  2 | 훈련 손실 4.7364


  에포크  3 | 훈련 손실 4.3321


  에포크  4 | 훈련 손실 4.0264


...

  에포크 11 | 훈련 손실 2.5752


  에포크 12 | 훈련 손실 2.4188


  에포크 13 | 훈련 손실 2.2698


  에포크 14 | 훈련 손실 2.1306


  에포크 15 | 훈련 손실 1.9998


In [8]:
PRIMER = ('dorothy , lion and scarecrow really worried about the emerald city , '
          'but oz said the city is safe because')
print('[정상 마중물]')
print(' ', generate_text(oz_writer, PRIMER, 30, oz_vocab, oz_reversed_vocab))
print()
print('[예외 처리 확인 1 — 어휘 사전에 없는 토큰이 섞인 마중물]')
print(' ', generate_text(oz_writer, 'dorothy met a quokka in kansas', 20, oz_vocab, oz_reversed_vocab))
print()
print('[예외 처리 확인 2 — 마중물이 짧은 경우]')
print(' ', generate_text(oz_writer, 'dorothy', 20, oz_vocab, oz_reversed_vocab))
print()
print('[예외 처리 확인 3 — 아는 토큰이 하나도 없는 경우]')
try:
    generate_text(oz_writer, 'quokka wombat numbat', 20, oz_vocab, oz_reversed_vocab)
except ValueError as error:
    print(f'  ValueError: {error}')

[정상 마중물]
  dorothy , lion and scarecrow really worried about the emerald city , but oz said the city is safe because you are . the scarecrow and the tin woodman must have to eat the great oz dwelt . the scarecrow was both pleased to be at home to pieces .

[예외 처리 확인 1 — 어휘 사전에 없는 토큰이 섞인 마중물]
  (주의) 어휘 사전에 없는 토큰 ['quokka'] -> 건너뜀
  dorothy met a in kansas , said the scarecrow , who are you , and why do you seek me ? i am oz ,

[예외 처리 확인 2 — 마중물이 짧은 경우]
  dorothy , said the scarecrow , but i will give you a heart , said the scarecrow . i have never

[예외 처리 확인 3 — 아는 토큰이 하나도 없는 경우]
  ValueError: 마중물 텍스트에 어휘 사전에 있는 토큰이 하나도 없습니다.


### 풀이 해설

**세 요구 사항은 각각 앞 장의 어떤 내용에 대응하는가**

| 요구 사항 | 배운 곳 |
|---|---|
| 지정한 장치를 사용하는 학습과 예측 | **5장** [코드 5-17], [코드 5-18] — `device` 인자를 받는 학습 함수 |
| 순환 신경망 모델의 학습 루프 | **6장** [코드 6-12] — 기울기 클리핑을 포함한 1 에포크 학습 함수 |
| 예외 처리를 갖춘 생성 함수 | **6장** [코드 6-8] — 자기회귀 생성 + 연습 문제 6-5에서 만난 예외 |

즉 이 문제는 새로운 것을 요구하지 않는다. **5장과 6장에서 따로 배운 세 가지를 한데 묶는 연습**이다.

**장치 지정에서 주의할 점**은 5장 p37~38에서 다룬 그대로다.
모델은 `to(device)`가 제자리 연산이지만 **텐서는 그렇지 않으므로** `inputs = inputs.to(device)`처럼
반환값을 다시 받아야 한다. 생성 함수에서도 마중물 텐서를 모델과 같은 장치로 보내야 한다.

**학습 루프에서 6장과 달라진 점은 없다.** 다만 기울기 클리핑은 넣어 두는 것이 좋다.
각주 9가 알려 주듯 LSTM도 기울기 폭주에서 완전히 자유롭지는 않다.

**예외 처리가 이 문제의 핵심이다.** 지문이 "비정상 종료되지 않도록"이라고 못 박은 이유가 있다.
생성 함수는 **사용자가 아무 문자열이나 넣을 수 있는 입구**다. 학습 루프와 달리 입력을 통제할 수 없다.
위 구현에서 처리한 세 가지는 다음과 같다.

**(1) 어휘 사전에 없는 토큰** — 가장 흔한 경우다. `vocab[token]`을 그대로 쓰면 `KeyError`가 난다.
실행 결과에서 `quokka`가 그 예다. `<unk>` 토큰이 있으면 그 번호로 대체하고, 없으면 건너뛴다.
**건너뛴 사실을 사용자에게 알려 주는 것**도 중요하다. 조용히 무시하면 결과가 왜 이상한지 알 수 없다.

**(2) 마중물이 윈도우보다 짧은 경우** — `'dorothy'` 한 단어만 넣어도 동작해야 한다.
앞을 마침표(문장 경계)로 채워 길이를 맞췄다. 6장 그림 6-15의 `<pad>` 발상과 같다.

**(3) 쓸 수 있는 토큰이 하나도 없는 경우** — 이때는 **조용히 넘어가면 안 된다.**
무엇을 생성해야 할지 알 수 없으므로 `ValueError`로 분명하게 알리는 것이 옳다.
**'예외 처리'가 '모든 예외를 삼키는 것'은 아니라는 점**이 이 세 번째 경우에서 드러난다.

**생성 결과**를 보면 마중물의 맥락을 이어받아 그럴듯한 문장을 만들어 낸다.
다만 15 에포크로는 본문(30 에포크)만큼 문장이 다듬어지지 않는다.

### 문제 검토

- **적절성: 적합. 복습 문제로 잘 설계되었다.** 7-2절 본문이 학습 루프와 생성 함수의 코드를 싣지 않고
  "이 과정의 코드 예제는 생략한다"(p13)고 넘어가는데, 이 문제가 그 빈자리를 채운다.
  본문이 생략한 것을 연습 문제가 받는 구조라 낭비가 없다.
- **[검토] 세 요구 사항을 항목으로 나열한 것이 좋다.** '학습 루프와 생성 함수를 만들어 보자'만 있었다면
  무엇을 챙겨야 할지 알기 어렵다. 항목으로 나누니 **자기 점검 목록**이 된다.
- **★ [검토] 세 번째 항목의 '예외 처리'가 이 문제에서 가장 값어치 있는 부분인데,
  어떤 예외를 말하는지 힌트가 없다.** 독자는 보통 '어휘 사전에 없는 토큰'만 떠올리는데,
  실제로는 **마중물이 윈도우보다 짧은 경우**도 있고, **쓸 수 있는 토큰이 하나도 없는 경우**도 있다.
  특히 마지막은 **예외를 삼키지 말고 오히려 일으켜야 하는 경우**라 배울 것이 많다.
  "마중물 텍스트에 어떤 문제가 있을 수 있을지 먼저 헤아려 보자" 정도를 덧붙이면 방향이 잡힌다.
- **[검토] '예제 코드를 확인하기 전 직접 테스트해 보자'는 안내가 친절하다.** 막히면 볼 곳이 있다는 안심을
  주면서도 먼저 해 보라고 권한다. 연습 문제 6-9와 같은 방식이라 책 전체의 태도가 일관된다.

**윤문안**

> **7-4** `OzWriter` 모델의 학습 루프와 문장 생성 함수를 직접 만들어 보자. 참고로 7-2절의 깃허브 예제 노트북을
> 조금만 내리면 저자가 작성한 예제 코드가 있지만, 이를 확인하기 전 5장과 6장에서 학습한 다음의 기능을
> 구현할 수 있는지 직접 테스트해 보자.
> - 지정한 CPU 또는 하드웨어 가속기 장치를 사용하는 학습과 예측
> - 순환 신경망 모델의 학습 루프
> - 정해진 수의 토큰을 생성해 문장을 만드는 생성 함수. 단, 비정상 종료되지 않도록 예외 처리가 되어 있어야 한다.
>   어떤 마중물 텍스트가 들어올 수 있을지 먼저 헤아려 본 후, 각각을 어떻게 처리할지 정해 구현해 보자.

## 연습 문제 7-5

> 소설 <오즈의 마법사>의 데이터셋을 만들 때, 전체의 10% 분량을 검증 데이터셋으로 분리해 모델을 학습시켜 보자.
> 에포크별 훈련 손실과 검증 손실을 확인하고 다음 질문에 답해 보자.
> - 과적합 여부를 확인할 수 있는가?
> - 확인 결과를 바탕으로 검증 손실과 평가 데이터셋으로 계산한 지표로 생성 모델의 성능 확인이 힘든 이유를 설명해 보자.

In [9]:
# 6장 p35의 주의를 따라, 슬라이딩 윈도우를 적용하기 전 토큰 단계에서 9:1로 나눈다
split_at = int(len(oz_idxs) * 0.9)
train_idxs, valid_idxs = oz_idxs[:split_at], oz_idxs[split_at:]

train_dataset = OzDataset(train_idxs, SEQUENCE_LENGTH)
valid_dataset = OzDataset(valid_idxs, SEQUENCE_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f'훈련 샘플 {len(train_dataset):,}개, 검증 샘플 {len(valid_dataset):,}개')

torch.manual_seed(SEED)
split_model = OzWriter(len(oz_vocab), EMBEDDING_DIM, HIDDEN_SIZE)
print('훈련/검증 분리 학습')
split_result = train(split_model, train_loader, valid_loader)

훈련 샘플 40,233개, 검증 샘플 4,453개
훈련/검증 분리 학습


  에포크  1 | 훈련 손실 5.6036 | 검증 손실 5.4036


  에포크  2 | 훈련 손실 4.7931 | 검증 손실 5.1678


  에포크  3 | 훈련 손실 4.3966 | 검증 손실 5.0406


  에포크  4 | 훈련 손실 4.0843 | 검증 손실 5.0063


...

  에포크 11 | 훈련 손실 2.6167 | 검증 손실 5.3477


  에포크 12 | 훈련 손실 2.4556 | 검증 손실 5.4444


  에포크 13 | 훈련 손실 2.3053 | 검증 손실 5.5272


  에포크 14 | 훈련 손실 2.1621 | 검증 손실 5.6216


  에포크 15 | 훈련 손실 2.0273 | 검증 손실 5.7246


In [10]:
print(f'{"에포크":>6} {"훈련 손실":>10} {"검증 손실":>10} {"차이":>9}')
print('-' * 40)
best_epoch, best_loss = 0, float('inf')
for epoch, train_loss, valid_loss in split_result['history']:
    if valid_loss < best_loss:
        best_loss, best_epoch = valid_loss, epoch
    print(f'{epoch:6d} {train_loss:10.4f} {valid_loss:10.4f} {valid_loss - train_loss:+9.4f}')
print()
print(f'검증 손실이 가장 낮은 에포크: {best_epoch} ({best_loss:.4f})')

   에포크      훈련 손실      검증 손실        차이
----------------------------------------
     1     5.6036     5.4036   -0.2000
     2     4.7931     5.1678   +0.3747
     3     4.3966     5.0406   +0.6440
     4     4.0843     5.0063   +0.9220
     5     3.8182     5.0062   +1.1880
     6     3.5800     5.0225   +1.4425
     7     3.3621     5.0501   +1.6880
     8     3.1580     5.1151   +1.9571
     9     2.9664     5.2096   +2.2433
    10     2.7852     5.2852   +2.4999
    11     2.6167     5.3477   +2.7310
    12     2.4556     5.4444   +2.9888
    13     2.3053     5.5272   +3.2219
    14     2.1621     5.6216   +3.4595
    15     2.0273     5.7246   +3.6973

검증 손실이 가장 낮은 에포크: 5 (5.0062)


In [11]:
# 검증 손실이 가장 낮을 때와 끝까지 학습했을 때의 생성 결과를 견줘 본다
print(f'[{EPOCHS} 에포크까지 학습한 모델의 생성 결과]')
print(' ', generate_text(split_model, PRIMER, 30, oz_vocab, oz_reversed_vocab))
print()
print('[같은 모델, 확률적 샘플링(temperature=0.7)]')
for _ in range(2):
    print(' ', generate_text(split_model, PRIMER, 30, oz_vocab, oz_reversed_vocab, temperature=0.7))

[15 에포크까지 학습한 모델의 생성 결과]
  dorothy , lion and scarecrow really worried about the emerald city , but oz said the city is safe because the scarecrow and the scarecrow and the tin woodman , and the lion said to the scarecrow and the lion and the lion and the tin woodman and the lion

[같은 모델, 확률적 샘플링(temperature=0.7)]
  dorothy , lion and scarecrow really worried about the emerald city , but oz said the city is safe because the cowardly lion , and they started on forever , for they had taken her so long before the witch she had done before . the figure was the winkies


  dorothy , lion and scarecrow really worried about the emerald city , but oz said the city is safe because the lion , and then , as the poor scarecrow was the great beast still back into the great forest , as soon as they came to the castle until


### 풀이 해설

**데이터 분리 방법부터 짚어야 한다**

6장 p35가 경고한 대로, **슬라이딩 윈도우를 적용한 뒤 무작위로 나누면 안 된다.**
이웃한 샘플이 19개 토큰을 공유하므로 훈련과 검증에 같은 정보가 중복된다.
여기서는 **토큰 리스트 단계에서 앞 90%, 뒤 10%로 잘랐다.**

어휘 사전은 전체에서 만들어 공유했다. 소설 뒷부분에만 나오는 단어가 있을 수 있는데,
훈련 데이터로만 사전을 만들면 검증 단계에서 `KeyError`가 난다(6장 p35).

**첫 번째 물음: 과적합 여부를 확인할 수 있는가**

**확인할 수 있다.** 위 표에서 검증 손실이 어느 지점부터 다시 올라가는 것을 볼 수 있고,
훈련 손실과 검증 손실의 차이가 계속 벌어진다. 4장에서 배운 과적합의 전형적인 모습 그대로다.

본문 p14가 `OzRawWriter`에 대해 "25 에포크에 이르러서는 `OzRawWriter`의 손실이 더 낮아졌다.
… 오히려 `OzRawWriter`가 학습 데이터에 과적합되기 시작했다는 신호다"라고 한 것이,
검증 손실을 재 보면 **추측이 아니라 사실로 확인된다.**

**두 번째 물음: 그런데도 왜 생성 모델의 성능 확인이 힘든가**

여기가 이 문제의 핵심이다. **검증 손실은 과적합은 알려 주지만 생성 품질은 알려 주지 않는다.**
이유는 세 가지다.

**(1) 검증 손실이 재는 것은 '원문과 얼마나 같은가'다.** 생성 모델이 잘하는 것은 '그럴듯한 새 문장을 만드는 것'이다.
본문 p18의 예처럼, '오늘 날씨가 정말' 다음에 원문이 '좋아'였는데 모델이 '맑아'를 냈다면
**손실은 커지지만 생성 품질은 나쁘지 않다.** 손실은 오히려 **정답과 다른 좋은 답에 벌점을 준다.**

**(2) 검증 손실은 '한 토큰 앞'만 본다.** 손실 계산은 늘 정답 문맥을 입력으로 주고 다음 한 토큰을 맞히게 한다.
그런데 실제 생성은 **자기가 만든 토큰을 다시 입력으로 쓴다.** 본문 p18이 지적한 대로
한 번 어긋나면 그 어긋남이 다음 예측의 입력이 되어 **오류가 연쇄적으로 커진다.**
손실은 이 연쇄를 전혀 측정하지 않는다. 그래서 **손실이 낮아도 길게 생성하면 무너질 수 있다.**

**(3) 정답이 하나가 아니다.** 분류 문제에서 고양이 사진의 정답은 고양이 하나뿐이지만,
'다음에 올 단어'의 정답은 여럿이다. **정답이 여럿인 문제를 정답 하나 기준으로 채점**하는 셈이다.

**그렇다면 검증 손실은 쓸모없는가.** 그렇지 않다. 위에서 확인했듯 **과적합 시점을 잡는 데는 유용하다.**
본문 p18의 표현대로 "학습이 잘 진행되고 있는지 또는 학습이 잘 끝났는지를 보여 주는 **보조 지표**"다.
최종 판단은 **생성 결과를 직접 읽어 보는 정성 평가**로 해야 한다.

위 마지막 셀에서 같은 모델로 세 번 생성해 보았다. `argmax` 방식은 늘 같은 문장을 내지만
확률적 샘플링을 쓰면 매번 다른 문장이 나온다. **같은 손실값의 모델이 전혀 다른 결과를 낼 수 있다**는 점도
손실만으로 생성 모델을 평가할 수 없는 이유 중 하나다.

### 문제 검토

- **적절성: 적합. 두 물음의 연결이 뛰어나다.** 첫 번째 물음('과적합을 확인할 수 있는가')의 답은 '예'인데,
  두 번째 물음은 그럼에도 '성능 확인은 힘들다'를 설명하게 한다. **'확인되는 것'과 '확인되지 않는 것'을**
  가르는 물음이라, 본문 p18의 정성 평가 설명을 제대로 소화했는지 드러난다.
- **[검토] 두 번째 물음의 "확인 결과를 바탕으로"가 좋다.** 첫 번째 물음의 실험 결과를 근거로 삼게 해
  두 물음이 따로 놀지 않는다.
- **★ [검토] 데이터 분리 방법에 대한 단서가 필요하다.** 6장 p35가 "슬라이딩 윈도우를 적용해 만든 데이터셋을
  `random_split()` 함수 등으로 무작위로 분리하면 제대로 된 평가가 되지 않는다"고 경고했는데,
  이 문제는 "전체의 10% 분량을 검증 데이터셋으로 분리해"라고만 한다.
  **'전체'가 토큰 리스트인지 슬라이딩 윈도우 적용 후의 데이터셋인지 모호하다.**
  6장의 주의를 잊은 독자는 `random_split()`을 쓸 텐데, 그러면 **검증 손실이 부풀려져 과적합이 잘 보이지 않고
  첫 번째 물음의 답이 '확인하기 어렵다'로 뒤집힌다.** 이 문제 전체가 무너지는 셈이다.
  → "6장에서 설명한 방법으로" 한마디만 넣으면 해결된다.
- **[검토] 정성 평가와 이어 주면 좋겠다.** 두 번째 물음의 답이 '그러니 생성 결과를 직접 봐야 한다'인데,
  실제로 생성해 보게 하지는 않는다. 검증 손실이 가장 낮은 모델과 끝까지 학습한 모델의 생성 결과를
  견주어 보게 하면 **손실과 품질이 따로 논다**는 것을 눈으로 확인할 수 있다.

**윤문안**

> **7-5** 소설 <오즈의 마법사>의 데이터셋을 만들 때, **6장에서 설명한 방법으로** 전체의 10% 분량을
> 검증 데이터셋으로 분리해 모델을 학습시켜 보자. 에포크별 훈련 손실과 검증 손실을 확인하고 다음 질문에 답해 보자.
> - 과적합 여부를 확인할 수 있는가?
> - 확인 결과를 바탕으로 검증 손실과 평가 데이터셋으로 계산한 지표로 생성 모델의 성능 확인이 힘든 이유를 설명해 보자.
> - 검증 손실이 가장 낮았던 시점의 모델과 끝까지 학습한 모델로 각각 문장을 생성해 비교해 보자.

## 연습 문제 7-6

> 임베딩 계층을 사용하는 `OzWriter` 모델은 `OzRawWriter` 모델에 비해 학습 속도가 빠르다.
> 이는 일반적인 현상인데, 그 이유를 데이터 처리 방식과 모델이 데이터를 이해하는 방식으로 나누어 제시해 보자.

In [12]:
torch.manual_seed(SEED)
oz_raw_writer = OzRawWriter(len(oz_vocab), HIDDEN_SIZE)
print(f'OzWriter    파라미터 수: {count_parameters(oz_writer):>10,d}')
print(f'OzRawWriter 파라미터 수: {count_parameters(oz_raw_writer):>10,d}')
print()
print('OzRawWriter 학습')
raw_result = train(oz_raw_writer, oz_loader, label='[원-핫] ')

OzWriter    파라미터 수:    894,358
OzRawWriter 파라미터 수:  1,967,766

OzRawWriter 학습


  [원-핫] 에포크  1 | 훈련 손실 5.9129


  [원-핫] 에포크  2 | 훈련 손실 5.3107


  [원-핫] 에포크  3 | 훈련 손실 4.8026


  [원-핫] 에포크  4 | 훈련 손실 4.4682


...

  [원-핫] 에포크 11 | 훈련 손실 3.0403


  [원-핫] 에포크 12 | 훈련 손실 2.8593


  [원-핫] 에포크 13 | 훈련 손실 2.6790


  [원-핫] 에포크 14 | 훈련 손실 2.5012


  [원-핫] 에포크 15 | 훈련 손실 2.3242


In [13]:
print(f'{"모델":>14} {"파라미터 수":>13} {"최종 훈련 손실":>14} {"전체 학습 시간":>14} {"에포크당":>10}')
print('-' * 74)
for name, model, result in [('OzWriter(임베딩)', oz_writer, oz_result),
                            ('OzRawWriter(원-핫)', oz_raw_writer, raw_result)]:
    print(f'{name:>14} {count_parameters(model):13,d} {result["history"][-1][1]:14.4f} '
          f'{result["elapsed"]:12.1f}초 {result["elapsed"] / EPOCHS:9.1f}초')

print()
print('에포크별 훈련 손실 비교')
print(f'{"에포크":>6} {"OzWriter":>10} {"OzRawWriter":>13} {"차이":>9}')
print('-' * 44)
for (epoch, oz_loss, _), (_, raw_loss, _) in zip(oz_result['history'], raw_result['history']):
    print(f'{epoch:6d} {oz_loss:10.4f} {raw_loss:13.4f} {raw_loss - oz_loss:+9.4f}')

            모델        파라미터 수       최종 훈련 손실       전체 학습 시간       에포크당
--------------------------------------------------------------------------
 OzWriter(임베딩)       894,358         1.9998         28.7초       1.9초
OzRawWriter(원-핫)     1,967,766         2.3242         27.3초       1.8초

에포크별 훈련 손실 비교
   에포크   OzWriter   OzRawWriter        차이
--------------------------------------------
     1     5.5581        5.9129   +0.3548
     2     4.7364        5.3107   +0.5743
     3     4.3321        4.8026   +0.4705
     4     4.0264        4.4682   +0.4419
     5     3.7652        4.2110   +0.4459
     6     3.5301        3.9905   +0.4605
     7     3.3128        3.7873   +0.4744
     8     3.1117        3.5981   +0.4864
     9     2.9213        3.4094   +0.4881
    10     2.7428        3.2263   +0.4835
    11     2.5752        3.0403   +0.4651
    12     2.4188        2.8593   +0.4405
    13     2.2698        2.6790   +0.4092
    14     2.1306        2.5012   +0.3706
    15     1.9998        

In [14]:
# 입력 텐서의 크기 차이를 직접 확인한다
sample_batch = next(iter(oz_loader))[0]
print(f'데이터로더가 내보내는 입력 텐서: {tuple(sample_batch.shape)}, {sample_batch.dtype}')
print(f'  -> 메모리: {sample_batch.element_size() * sample_batch.numel() / 1024:.1f} KB')

onehot_batch = nn.functional.one_hot(sample_batch, num_classes=len(oz_vocab)).float()
print(f'원-핫 인코딩 후: {tuple(onehot_batch.shape)}, {onehot_batch.dtype}')
print(f'  -> 메모리: {onehot_batch.element_size() * onehot_batch.numel() / 1024**2:.1f} MB')
print(f'  -> 배율: {onehot_batch.element_size() * onehot_batch.numel() / (sample_batch.element_size() * sample_batch.numel()):.0f}배')

데이터로더가 내보내는 입력 텐서: (64, 20), torch.int64
  -> 메모리: 10.0 KB
원-핫 인코딩 후: (64, 20, 2966), torch.float32
  -> 메모리: 14.5 MB
  -> 배율: 1483배


### 풀이 해설

지문이 요구한 두 관점으로 나누어 답한다. 다만 **먼저 실측 결과를 정직하게 봐야 한다.**

| 모델 | 파라미터 수 | 최종 훈련 손실 | 전체 학습 시간 | 에포크당 |
|---|---|---|---|---|
| OzWriter (임베딩) | 894,358 | **1.9998** | 28.7초 | 1.9초 |
| OzRawWriter (원-핫) | 1,967,766 | 2.3242 | **27.3초** | **1.8초** |

**손실은 `OzWriter`가 일관되게 낮지만, 학습 시간은 오히려 `OzRawWriter`가 미세하게 빠르게 나왔다.**
본문 각주 9의 측정(53초 대 203초)과 상당히 다르다. 이 차이가 왜 생기는지는 아래에서 다시 다룬다.

**관점 1: 데이터 처리 방식 — 계산량과 메모리가 적다**

`OzRawWriter`는 토큰마다 **어휘 사전 크기(2,966)만큼 긴 원-핫 벡터**를 만들어 LSTM에 넣는다.
위 실행 결과에서 확인한 대로, 같은 배치가 원-핫으로 바뀌면 **10KB에서 14.5MB로 약 1,483배**가 된다.

이것이 세 가지 비용으로 이어진다.

- **메모리와 전송**: 배치마다 14.5MB짜리 텐서를 만들고 옮겨야 한다.
- **LSTM 계층의 크기**: `input_size`가 2,966이 되어 입력 쪽 가중치 행렬이 그만큼 커진다.
  실제로 `OzRawWriter`의 파라미터가 **약 2.2배**다.
- **낭비되는 곱셈**: 원-핫 벡터는 1이 하나, 나머지 2,965개가 0이다. 그런데 행렬곱은 0인 자리까지 전부 곱한다.
  **계산의 99.97%가 0을 곱해 0을 더하는 일**이다.

반면 `OzWriter`는 **행렬곱 대신 인덱싱**으로 임베딩 벡터를 꺼내고(본문 p5),
LSTM의 `input_size`가 128로 줄어 계층 자체가 작아진다.

**그런데 왜 시간 차이가 나지 않았는가**

이 실험에서는 **원-핫 인코딩을 모델의 `forward()` 안에서, GPU 위에서** 수행했다.
그러면 큰 텐서가 GPU 메모리 안에서만 만들어지고 사라지므로 전송 비용이 들지 않고,
cuDNN이 행렬 연산을 병렬로 처리해 0을 곱하는 낭비도 잘 드러나지 않는다.
게다가 순환 신경망의 병목은 연산량이 아니라 **20개 시점을 순서대로 처리해야 하는 구조**에 있다.

**만약 원-핫 인코딩을 데이터셋이나 CPU에서 수행했다면 이야기가 달라진다.**
배치마다 14.5MB를 CPU에서 만들어 GPU로 보내야 하므로, 이 전송이 학습 시간을 지배한다.
본문 각주 9의 **53초 대 203초(약 4배)**는 이 경우에 가까워 보인다.

즉 **'임베딩이 빠르다'는 참이지만, 얼마나 빠른지는 원-핫 인코딩을 어디서 하느냐에 크게 좌우된다.**
파라미터 수와 메모리 크기라는 **구조적 이점은 환경과 무관하게 성립**하지만,
시계로 잰 시간은 환경에 따라 달라진다.

**관점 2: 모델이 데이터를 이해하는 방식 — 배울 것이 줄어든다**

이쪽이 더 근본적이고, **이번 실험에서 실제로 확인된 쪽**이다.

원-핫 인코딩에서는 **모든 토큰이 서로 완전히 독립**이다(연습 문제 7-1에서 확인했듯 어떤 두 토큰이든
거리가 √2로 같다). 그래서 `OzRawWriter`는 `witch`에서 배운 것을 `wizard`에 전혀 활용하지 못한다.
**단어 하나하나의 쓰임을 따로따로 외워야 한다.**

반면 `OzWriter`의 임베딩 계층은 **비슷한 역할을 하는 단어를 비슷한 벡터로 모은다.**
그러면 LSTM 입장에서는 3천 개의 서로 다른 입력이 아니라 **몇 가지 부류**로 정리된 입력을 받는 셈이다.
본문 p14가 "임베딩 계층이 토큰 사이의 관계를 앞에서 미리 정리해 준 덕분에 연결된 LSTM 계층이
좀 더 효율적으로 순서 특징에 집중할 수 있다"고 한 것이 이 뜻이다.

**비유하자면** 원-핫 방식은 학생에게 단어 카드 3천 장을 낱장으로 주고 전부 외우게 하는 것이고,
임베딩 방식은 카드를 **주제별로 묶어** 준 다음 문장을 가르치는 것이다.

위 표의 에포크별 비교에서 **`OzWriter`의 손실이 1 에포크부터 끝까지 일관되게 낮다.**
**'같은 에포크에서 더 많이 배운다'**는 뜻이며, 이것이 '학습 속도가 빠르다'의 더 정확한 의미다.

**덧붙임**: 본문 p14가 지적했듯 **학습이 길어지면 `OzRawWriter`의 손실이 더 낮아지는 구간이 온다.**
그것은 성능이 좋아진 것이 아니라 **외우기 시작한 것**이다.
원-핫 방식은 토큰을 독립적으로 다루므로 **개별 등장 패턴을 외우기가 오히려 쉽다.**
이 실험은 15 에포크에서 끝나 그 역전 지점(본문 기준 25 에포크)까지는 가지 않았다.

### 문제 검토

- **적절성: 적합. 관점을 둘로 나누라고 지정한 것이 이 문제의 핵심이다.**
  그냥 '이유를 설명하라'고 하면 대부분 '벡터가 짧아서 계산이 빨라진다'는 **데이터 처리 쪽 답 하나**로 끝난다.
  '모델이 데이터를 이해하는 방식'이라는 두 번째 관점을 못 박아 두어서, 본문 p14가 말한
  "임베딩 계층이 토큰 사이의 관계를 앞에서 미리 정리해 준 덕분에"를 자기 말로 풀어내게 만든다.
  **7장에서 가장 잘 만든 물음 중 하나다.**
- **★ [검토] 각주 9의 측정값(53초 대 203초)이 환경에 따라 재현되지 않을 수 있다.**
  실제 측정 근거를 실어 둔 것 자체는 좋지만, **CUDA GPU에서 원-핫 인코딩을 모델의 `forward()` 안에서 수행하면
  두 모델의 학습 시간이 거의 같게 나온다**(이 노트북의 실측: 28.7초 대 27.3초).
  약 4배 차이는 원-핫 텐서를 **CPU에서 만들어 GPU로 옮길 때** 주로 나타나는데,
  배치마다 14.5MB를 전송해야 하기 때문이다.
  독자가 자기 결과와 견주다가 "왜 안 빨라지지?" 하고 막힐 수 있으므로,
  **각주에 측정 환경을 한마디 덧붙이거나 '환경에 따라 차이가 달라질 수 있다'를 밝히면** 좋겠다.
  본문 주장의 **구조적 근거(파라미터 수, 메모리 크기, 같은 에포크에서의 손실)는 환경과 무관하게 성립**하므로
  본문 서술 자체를 바꿀 필요는 없다.
- **[검토] '일반적인 현상'이라고 미리 밝힌 것도 정확하다.** 이 예제 하나의 우연이 아니라는 점을 알려 준다.
- **[검토] 실측을 요구하지 않는 것은 이 문제에서는 적절하다.** 7-4, 7-5가 이미 구현과 실행을 요구했으므로,
  이 문제는 **개념을 정리하는 자리**로 두는 편이 절 전체의 리듬에 맞는다.

**결론: 문제 지문 자체는 수정할 것이 없다. 다만 각주 9에 측정 환경을 밝히는 것을 권한다.**

## 연습 문제 7-7

> 소설 <오즈의 마법사>가 미국을 대표하는 환상 소설 중 하나라면, 루이스 캐럴이 1865년 발표한
> <이상한 나라의 앨리스>는 영국을 대표하는 환상 소설 중 하나이다. 이 작품 역시 구텐베르크 프로젝트에서
> 저작권 없는 텍스트 파일로 내려받을 수 있다. 파일을 내려받아 전처리한 다음 소설 <오즈의 마법사>의 데이터와
> 합쳐 `OzWriter` 모델을 학습한 뒤, 두 소설에 나오는 문자열을 포함해 다양한 마중물 텍스트의 생성 결과를 확인해 보자.

In [15]:
alice_tokens = tokenize(load_novel(f'{DATA_DIR}/alice_in_wonderland.txt'))
print(f'<이상한 나라의 앨리스> 전체 토큰 {len(alice_tokens):,}개, 고유 토큰 {len(set(alice_tokens)):,}개')

oz_set, alice_set = set(oz_tokens), set(alice_tokens)
print()
print(f'두 소설에 모두 나오는 단어: {len(oz_set & alice_set):,}개')
print(f'<오즈>에만 나오는 단어   : {len(oz_set - alice_set):,}개')
print(f'<앨리스>에만 나오는 단어 : {len(alice_set - oz_set):,}개')
print(f'합친 어휘 사전의 크기    : {len(oz_set | alice_set):,}개')
print()
print(f'<오즈>에만 나오는 단어 예시: {sorted(oz_set - alice_set)[:12]}')
print(f'<앨리스>에만 나오는 단어 예시: {sorted(alice_set - oz_set)[:12]}')

<이상한 나라의 앨리스> 전체 토큰 30,857개, 고유 토큰 2,640개

두 소설에 모두 나오는 단어: 1,375개
<오즈>에만 나오는 단어   : 1,591개
<앨리스>에만 나오는 단어 : 1,265개
합친 어휘 사전의 크기    : 4,231개

<오즈>에만 나오는 단어 예시: ['abounding', 'abundance', 'accidents', 'ached', 'action', 'add', 'addition', 'admit', 'admitted', 'advanced', 'adventure', 'afresh']
<앨리스>에만 나오는 단어 예시: ['abide', 'absence', 'absurd', 'acceptance', 'accidentally', 'accounting', 'accusation', 'accustomed', 'ache', 'ada', 'adding', 'addressed']


In [16]:
# 두 소설을 이어 붙인다. 소설 경계를 넘나드는 샘플이 생기지 않도록 따로 데이터셋을 만든 뒤 합친다
from torch.utils.data import ConcatDataset

both_tokens = oz_tokens + alice_tokens
both_vocab, both_reversed_vocab = build_vocab(both_tokens)
oz_both_idxs = [both_vocab[t] for t in oz_tokens]
alice_both_idxs = [both_vocab[t] for t in alice_tokens]

both_dataset = ConcatDataset([OzDataset(oz_both_idxs, SEQUENCE_LENGTH),
                             OzDataset(alice_both_idxs, SEQUENCE_LENGTH)])
both_loader = DataLoader(both_dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f'합친 어휘 사전 크기: {len(both_vocab):,} (오즈만일 때 {len(oz_vocab):,})')
print(f'합친 샘플 수: {len(both_dataset):,} (오즈만일 때 {len(oz_dataset):,})')

torch.manual_seed(SEED)
both_model = OzWriter(len(both_vocab), EMBEDDING_DIM, HIDDEN_SIZE)
print()
print('두 소설을 합쳐 학습')
both_result = train(both_model, both_loader, label='[합본] ')

합친 어휘 사전 크기: 4,231 (오즈만일 때 2,966)
합친 샘플 수: 75,543 (오즈만일 때 44,706)

두 소설을 합쳐 학습


  [합본] 에포크  1 | 훈련 손실 5.5655


  [합본] 에포크  2 | 훈련 손실 4.7542


  [합본] 에포크  3 | 훈련 손실 4.3556


  [합본] 에포크  4 | 훈련 손실 4.0475


...

  [합본] 에포크 11 | 훈련 손실 2.6277


  [합본] 에포크 12 | 훈련 손실 2.4795


  [합본] 에포크 13 | 훈련 손실 2.3434


  [합본] 에포크 14 | 훈련 손실 2.2149


  [합본] 에포크 15 | 훈련 손실 2.0954


In [17]:
PRIMERS = {
    '오즈의 문장': 'the scarecrow and the tin woodman walked along the yellow brick road toward the emerald city',
    '앨리스의 문장': 'alice was beginning to get very tired of sitting by her sister on the bank and of having',
    '두 소설을 섞은 문장': 'dorothy and alice walked together toward the emerald city and the white rabbit said',
    '어느 쪽에도 없는 문장': 'the programmer opened a laptop and started to write a deep learning model',
}
for name, primer in PRIMERS.items():
    print(f'[{name}]')
    print(f'  마중물: {primer}')
    print(f'  생성  : {generate_text(both_model, primer, 25, both_vocab, both_reversed_vocab, temperature=0.7)}')
    print()

[오즈의 문장]
  마중물: the scarecrow and the tin woodman walked along the yellow brick road toward the emerald city
  생성  : the scarecrow and the tin woodman walked along the yellow brick road toward the emerald city . when this politeness found no one of the road , and these were be lovely patches in all the crows , or bow now

[앨리스의 문장]
  마중물: alice was beginning to get very tired of sitting by her sister on the bank and of having
  생성  : alice was beginning to get very tired of sitting by her sister on the bank and of having the wicked witch of the west . the witch of the east was angry , and she thought it was a great many about and

[두 소설을 섞은 문장]
  마중물: dorothy and alice walked together toward the emerald city and the white rabbit said
  생성  : dorothy and alice walked together toward the emerald city and the white rabbit said it . i don’t mind the emerald city to see the wicked witch i will carry you to kansas . these are all the witch

[어느 쪽에도 없는 문장]
  마중물: the programmer opened a lapto

### 풀이 해설

**두 소설을 어떻게 합칠 것인가**

여기에 **설계 판단이 하나 숨어 있다.** 6장 연습 문제 6-8에서 네 동요를 합칠 때와 같은 문제다.

- **토큰 리스트를 그냥 이어 붙이면**, 슬라이딩 윈도우가 **<오즈>의 마지막 문장과 <앨리스>의 첫 문장에
  걸친 샘플**을 만든다. 존재하지 않는 문맥을 학습하는 셈이다.
- **소설마다 따로 데이터셋을 만들어 `ConcatDataset`으로 합치면** 이 문제가 사라진다.

샘플 스무 개짜리 경계 오류가 전체에 미치는 영향은 작지만, **원리상 올바른 쪽을 택하는 것이 좋다.**

**어휘 사전이 크게 늘어난다**

| | 토큰 수 | 고유 토큰 |
|---|---|---|
| <오즈의 마법사> | 44,726 | 2,966 |
| <이상한 나라의 앨리스> | 30,857 | 2,640 |
| **합본** | **75,583** | **4,231** |

두 소설이 공유하는 단어는 1,375개뿐이고, <오즈>에만 1,591개, <앨리스>에만 1,265개가 있다.
**겹치는 어휘가 전체의 3분의 1도 안 된다.** 그래서 합치면 어휘 사전이 2,966에서 4,231로 43% 늘고,
**임베딩 행렬의 행 수와 완전 연결 계층의 출력 크기가 함께 늘어난다.**

**생성 결과 — 예상과 다른 일이 벌어진다**

마중물을 네 가지로 나누어 넣어 보았다.

```
[오즈의 문장]  the scarecrow and the tin woodman walked along the yellow brick road toward the emerald city
  -> . when this politeness found no one of the road , and these were be lovely patches in all the crows , or bow now

[앨리스의 문장] alice was beginning to get very tired of sitting by her sister on the bank and of having
  -> the wicked witch of the west . the witch of the east was angry , and she thought it was a great many about and

[두 소설을 섞은 문장] dorothy and alice walked together toward the emerald city and the white rabbit said
  -> it . i don’t mind the emerald city to see the wicked witch i will carry you to kansas . these are all the witch

[어느 쪽에도 없는 문장] the programmer opened a laptop and started to write a deep learning model
  (주의) 어휘 사전에 없는 토큰 ['programmer', 'laptop', 'model'] -> 건너뜀
  -> ... longer from surprise . i was going to the scarecrow and the wicked witch of the west . you want , said the lion .
```

**가장 눈에 띄는 것은 두 번째다.** `alice`로 시작하는 <앨리스>의 첫 문장을 그대로 넣었는데
**곧바로 `the wicked witch of the west`(<오즈>의 서쪽 마녀)로 넘어간다.**
`alice`도 `sister`도 `bank`도 <앨리스>의 단어인데, 모델은 <앨리스>의 세계를 이어 가지 못한다.

세 번째와 네 번째에서도 마찬가지로 **모든 생성이 <오즈> 쪽으로 기운다.**
`the white rabbit`(<앨리스>)로 끝나는 마중물에서도 `emerald city`, `wicked witch`, `kansas`가 나온다.

**왜 <오즈> 쪽으로 쏠리는가.** 이유는 단순하다. **<오즈>의 토큰이 44,726개로 <앨리스>(30,857개)보다 45% 많다.**
모델은 더 자주 본 쪽의 패턴을 더 강하게 배운다. 데이터의 양이 곧 목소리의 크기가 된 셈이다.

**이 문제에서 배울 것**

**데이터를 늘리는 것이 언제나 좋기만 한 것은 아니다.** 얻는 것과 잃는 것이 함께 있다.

| 얻는 것 | 잃는 것 |
|---|---|
| 샘플 수가 44,706개에서 75,543개로 늘어 학습 기회가 많아진다 | 어휘 사전이 43% 커져 모델이 무거워진다 |
| 영어 문법과 문체의 공통 패턴을 더 잘 배운다 | **두 세계가 섞이고, 양이 많은 쪽으로 쏠린다** |
| 처음 보는 단어가 줄어든다 | 각 소설만의 특징이 희석된다 |

**'<오즈의 마법사>처럼 쓰는 모델'을 원한다면 오히려 <오즈>만으로 학습하는 것이 낫다.**
'<이상한 나라의 앨리스>처럼 쓰는 모델'을 원한다면 합치는 것은 **분명히 손해**다.
실제로 위 결과에서 <앨리스>의 색채는 거의 남지 않았다.

두 소설을 합치는 것이 도움이 되는 경우는 **'영어 문장을 만드는 능력' 자체를 키우고 싶을 때**다.
그리고 그 방향으로 제대로 가려면 두 권이 아니라 **수만 권이 필요하다.**
이것이 대규모 언어 모델이 방대한 텍스트를 쓰는 이유이며, 본문 p15가 짚은 지점이다.

**연습 문제 7-15와의 관계**: 7-15는 같은 목표를 **전이 학습**으로 접근한다.
합치는 대신 <오즈>로 학습한 임베딩을 가져와 <앨리스>에 맞게 **미세 조정**하는 방식이다.
위에서 본 '양이 많은 쪽으로 쏠리는' 문제가 전이 학습에서는 어떻게 달라지는지 비교해 보면 좋다.

### 문제 검토

- **적절성: 적합. 7-2절을 마무리하는 문제로 알맞다.** 지금까지 만든 것을 **새 데이터에 적용**해 보게 하고,
  7-15(전이 학습)의 밑그림도 된다. <이상한 나라의 앨리스>를 고른 것도 좋다.
  <오즈의 마법사>와 시대·장르가 비슷해 비교 조건이 깨끗하고, 저작권 문제도 없다.
- **★ [검토] 두 소설을 '합치는' 방법에 갈림길이 있는데 지문이 이를 드러내지 않는다.**
  토큰 리스트를 그냥 이어 붙이면 **두 소설의 경계에 걸친 샘플**이 만들어진다.
  6장 연습 문제 6-8에서 네 동요를 합칠 때와 똑같은 문제이므로, 그 문제를 푼 독자라면 떠올릴 수도 있다.
  다만 이번에는 경계가 한 곳뿐이라 영향이 작아 **알아채지 못하고 넘어가기 더 쉽다.**
  한마디 덧붙이면 앞 장과의 연결도 살아난다.
- **★ [검토] '다양한 마중물 텍스트'가 너무 열려 있다.** 이 문제의 관찰거리는 **마중물의 출처에 따라 결과가
  어떻게 달라지는가**인데, '다양한'만으로는 독자가 아무 문장이나 몇 개 넣어 보고 끝낸다.
  네 가지로 나누어 제시하면 관찰이 훨씬 또렷해진다.
  (오즈의 문장 / 앨리스의 문장 / **두 소설을 섞은 문장** / 어느 쪽에도 없는 문장)
  실제로 돌려 보면 **<앨리스>의 첫 문장을 그대로 넣어도 곧바로 <오즈>의 서쪽 마녀 이야기로 넘어간다.**
  <오즈>의 토큰이 45% 더 많아 모델이 그쪽으로 쏠리기 때문인데, 마중물을 나누어 넣어 봐야 이것이 보인다.
- **★ [검토] '합치면 좋아지는가'를 묻지 않는다.** 독자는 '데이터가 늘었으니 좋아지겠지'라고 기대하며 시작하는데,
  실제로는 **문체의 일관성이 오히려 흔들린다.** 이 대비가 이 문제의 실질적인 교훈인데 지문이 가리키지 않는다.
  <오즈>만으로 학습한 모델과 견주어 보게 하면 얻는 것과 잃는 것이 함께 드러난다.
- **[검토] URL을 지문에 실은 것은 친절하지만 저장소에도 파일이 있다.** 저장소의 `data/` 디렉터리에
  `alice_in_wonderland.txt`가 이미 들어 있으므로, 6-3절의 <오즈의 마법사> 안내처럼
  "또는 깃허브 저장소 `data` 디렉터리의 `alice_in_wonderland.txt` 파일을 사용해도 된다"를 덧붙이면
  내려받기가 막히는 독자도 실습할 수 있다.

**윤문안**

> **7-7** 소설 <오즈의 마법사>가 미국을 대표하는 환상 소설 중 하나라면, 루이스 캐럴^Lewis Carroll^이 1865년 발표한
> <이상한 나라의 앨리스^Alice's Adventures in Wonderland^>는 영국을 대표하는 환상 소설 중 하나이다. 이 작품 역시
> 구텐베르크 프로젝트에서 저작권 없는 텍스트 파일로 내려받을 수 있다(https://www.gutenberg.org/cache/epub/11/pg11.txt).
> **또는 깃허브 저장소 `data` 디렉터리의 `alice_in_wonderland.txt` 파일을 사용해도 된다.**
> 파일을 내려받아 전처리한 다음 소설 <오즈의 마법사>의 데이터와 합쳐 `OzWriter` 모델을 학습해 보자.
> **이때 두 소설을 어떻게 이어 붙여야 할지도 함께 생각해 보자.**
> 학습한 뒤 다음 네 종류의 마중물 텍스트로 생성 결과를 확인하고, <오즈의 마법사>만으로 학습한 모델과 비교해 보자.
> - <오즈의 마법사>에 나오는 문장
> - <이상한 나라의 앨리스>에 나오는 문장
> - 두 소설의 등장인물을 함께 넣은, 원문에 없는 문장
> - 두 소설 어디에도 나오지 않는 문장